# Crop-seq (sVAEplus) vs OPS Phase

Compare mean average precision (mAP) of the sVAEplus crop-seq embedding against
the OPS `cell_dino` Phase-only embedding on perturbations and protein complexes
that are present in both data modalities.

## Imports

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["svg.fonttype"] = "none"

FIGURES_DIR = Path("../../output/figure_5")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## CSV paths

Explicit path to every evaluation CSV used below.

**Before public release**, replace this cell with download instructions (or a pointer to the public dataset) and update the constants to match the released layout. Note: the crop-seq EBI CSV currently lives in an `ops_monorepo` scratch directory and will need to be moved before release.

In [ ]:
# Crop-seq sVAEplus distinctiveness (variant: std_ntc)
SVAEPLUS_DISTINCTIVENESS_STD_NTC = "/hpc/projects/data.science/duo.peng/sVAEplus/sVAEplus/6000HVG/svaeplus_results_2_256_1_200_0.5/mAP_distinctiveness_std_ntc.csv"

# OPS cell_dino All-Fluorescence (no_phase) \u2014 used to define the shared perturbation / complex sets
CELL_DINO_NO_PHASE_DISTINCTIVENESS = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/cell_dino/zscore_per_exp/paper_v1/no_phase/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_distinctiveness.csv"
CELL_DINO_NO_PHASE_EBI             = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/cell_dino/zscore_per_exp/paper_v1/no_phase/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_consistency_ebi.csv"

# OPS cell_dino Phase-only
CELL_DINO_PHASE_DISTINCTIVENESS    = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/cell_dino/zscore_per_exp/paper_v1/phase_only/fixed_80%/cosine/metrics/phenotypic_distinctiveness.csv"
CELL_DINO_PHASE_TITRATION          = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/cell_dino/zscore_per_exp/paper_v1/with_cp/with_4i/all_livecell/fixed_80%/cosine/titration_guide_median/Phase/Phase_titration.csv"

# Crop-seq EBI mAP (currently in ops_monorepo scratch; move before public release)
CROPSEQ_EBI                        = "/hpc/mydata/alexander.hillsley/ops/ops_monorepo/experiments/scratch/crop-seq/20260520_mAP_ebi_cropseq.csv"

## Crop-seq vs OPS Phase \u2014 shared perturbations / complexes

Restrict each metric to entries shared across data modalities and compute mean mAP.

- **Distinctiveness**: 3-way intersection of `perturbation` across crop-seq, OPS All-Fluorescence, and OPS Phase.
- **EBI**: 2-way intersection on `complex_num` between crop-seq and OPS All-Fluorescence; the OPS Phase value is the maximum across the phase titration (no per-complex CSV is available).

In [ ]:
cs_distinct = pd.read_csv(SVAEPLUS_DISTINCTIVENESS_STD_NTC)
ao_distinct = pd.read_csv(CELL_DINO_NO_PHASE_DISTINCTIVENESS)
p_distinct  = pd.read_csv(CELL_DINO_PHASE_DISTINCTIVENESS)

cs_ebi = pd.read_csv(CROPSEQ_EBI)
ao_ebi = pd.read_csv(CELL_DINO_NO_PHASE_EBI)
p_ebi  = pd.read_csv(CELL_DINO_PHASE_TITRATION)

rows = []

# distinctiveness: 3-way shared on "perturbation"
common = set(cs_distinct["perturbation"]) & set(ao_distinct["perturbation"]) & set(p_distinct["perturbation"])
cs_sub = cs_distinct[cs_distinct["perturbation"].isin(common)]
p_sub  = p_distinct[p_distinct["perturbation"].isin(common)]
rows.append({
    "metric": "distinctiveness", "n_shared": len(common),
    "crop-seq mean": cs_sub["mean_average_precision"].mean(),
    "phase mean":    p_sub["mean_average_precision"].mean(),
})

# ebi: 2-way shared on complex_num; phase from titration max
common = set(cs_ebi["complex_num"]) & set(ao_ebi["complex_num"])
cs_sub = cs_ebi[cs_ebi["complex_num"].isin(common)]
rows.append({
    "metric": "ebi", "n_shared": len(common),
    "crop-seq mean": cs_sub["mean_average_precision"].mean(),
    "phase mean":    p_ebi["ebi_map_mean"].max(),
})

cmp_df = pd.DataFrame(rows)
cmp_df

## Mean mAP \u2014 paper figure

Single-panel comparison of crop-seq vs OPS Phase on mean mAP, saved as SVG for the paper.

In [ ]:
metrics_order = ["distinctiveness", "ebi"]
x = range(len(metrics_order))
width = 0.27
COLORS = {"crop-seq": "#4C72B0", "phase": "#55A868"}

cs_vals = [cmp_df.loc[cmp_df["metric"] == m, "crop-seq mean"].values[0] for m in metrics_order]
p_vals  = [cmp_df.loc[cmp_df["metric"] == m, "phase mean"].values[0]    for m in metrics_order]
ns      = [cmp_df.loc[cmp_df["metric"] == m, "n_shared"].values[0]      for m in metrics_order]

fig, ax = plt.subplots(figsize=(5, 5))
bars_cs = ax.bar([i - width/2 for i in x], cs_vals, width, label="crop-seq",  color=COLORS["crop-seq"], edgecolor="white")
bars_p  = ax.bar([i + width/2 for i in x], p_vals,  width, label="OPS Phase", color=COLORS["phase"],    edgecolor="white")

for bars, vals in [(bars_cs, cs_vals), (bars_p, p_vals)]:
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(list(x))
ax.set_xticklabels([f"{m}\n(n={n})" for m, n in zip(metrics_order, ns)], fontsize=9)
ax.set_ylabel("Mean mAP")
ax.set_ylim(0, 0.6)
ax.yaxis.set_major_locator(mticker.MultipleLocator(0.2))
ax.grid(axis="y", linewidth=0.5, alpha=0.5)
ax.legend(loc="upper left")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "cropseq_vs_celldino_map.svg", bbox_inches="tight")
plt.show()